# الفصل 8: الشبكات المتكررة: RNN وLSTM وGRU

> قبل المحوّلات (2017)، كانت الشبكات المتكررة (Recurrent Neural Networks) هي المعمارية المهيمنة الوحيدة لمعالجة التسلسلات اللغوية.

- **الجزء:** 2 — التمثيل اللغوي
- **الفصل:** 8
- **المستوى:** متوسط
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 7: التمثيلات المتجهية للكلمات: Word2Vec وGloVe
- **الوسوم:** RNN، LSTM، GRU، BPTT

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** numpy, torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("numpy", "numpy"),
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-02-language-representation%2Fch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-02-language-representation/ch08-recurrent-networks.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> قبل المحوّلات (2017)، كانت الشبكات المتكررة (Recurrent Neural Networks) هي المعمارية المهيمنة الوحيدة لمعالجة التسلسلات اللغوية.

> **بيانات الفصل**
>
> - **الجزء:** 2 — التمثيل اللغوي
> - **المستوى:** متوسط
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 7: التمثيلات المتجهية للكلمات: Word2Vec وGloVe


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم LSTM — الحل عبر البوابات (Gates)
- فهم GRU — نسخة مُبسَّطة من LSTM
- فهم الانتقال لـ PyTorch ومعمارية Bidirectional


## 1. لماذا يهمّك هذا الموضوع؟

قبل المحوّلات (2017)، كانت الشبكات المتكررة (Recurrent Neural Networks) هي **المعمارية المهيمنة الوحيدة** لمعالجة التسلسلات اللغوية. فهمها بعمق — وتحديدًا **فهم أسباب فشلها** — هو ما يُفسِّر بدقة لماذا اختُرعت آلية Attention، ولماذا أحدثت المحوّلات ثورة حقيقية. بدون هذا الفهم التاريخي، ستتعامل مع Attention كـ"حيلة رياضية عشوائية" بدلًا من فهمها كـ**حل مباشر ومقصود** لمشاكل محددة وموثّقة جيدًا.


## 2. الخريطة الذهنية


```text
تطوّر معالجة التسلسلات:

RNN البسيطة          →     LSTM                →     GRU              →    Transformer
(1986/1990s)               (1997)                    (2014)                 (2017، الفصل 10)
     │                        │                         │                       │
معالجة متسلسلة          بوابات لحل مشكلة          تبسيط LSTM             معالجة متوازية
تذكّر قصير فقط          تلاشي التدرجات             (أقل معاملات)           بالكامل + Attention
(Vanishing Gradient)    (Memory Cell + Gates)                              يحل المشاكل جذريًا
```


## 3. الشرح المتدرج

### 🟢 طبقة الحدس — RNN البسيطة

تخيّل أنك تقرأ جملة كلمة كلمة، وتحتفظ في ذاكرتك بـ"ملخص" لكل ما قرأته حتى الآن (الحالة المخفية، Hidden State). عند كل كلمة جديدة، تُحدِّث هذا الملخص بدمج المعلومة الجديدة مع الملخص السابق. هذا بالضبط ما تفعله RNN: **نفس الأوزان** تُطبَّق بشكل متكرر عبر كل خطوة زمنية، مع تمرير حالة مخفية من خطوة لأخرى.

### 🟡 طبقة الآلية — معادلة RNN الأساسية


```yaml
h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b_h)
y_t = W_hy · h_t + b_y

حيث:
h_t: الحالة المخفية عند الزمن t (الذاكرة المتراكمة)
x_t: مدخل الزمن t (مثلًا embedding الكلمة الحالية)
W_hh, W_xh, W_hy: مصفوفات أوزان مُشتركة (نفسها) عبر كل الخطوات الزمنية
```


### 🔴 طبقة التعمّق — RNN من الصفر ومشكلة تلاشي التدرجات


In [ ]:
import numpy as np

class SimpleRNNCell:
    """
    تطبيق خلية RNN واحدة من الصفر، لتوضيح الآلية الأساسية
    قبل الانتقال لتطبيق PyTorch الكامل.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        self.hidden_dim = hidden_dim
        scale = 1.0 / np.sqrt(hidden_dim)
        self.W_xh = np.random.randn(input_dim, hidden_dim) * scale
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.b_h = np.zeros(hidden_dim)

    def forward_step(self, x_t: np.ndarray, h_prev: np.ndarray) -> np.ndarray:
        z = x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h
        h_t = np.tanh(z)
        return h_t

    def forward_sequence(self, X: np.ndarray) -> list:
        """معالجة تسلسل كامل، خطوة زمنية تلو الأخرى (معالجة متسلسلة إلزامية!)."""
        seq_len = X.shape[0]
        h = np.zeros(self.hidden_dim)
        hidden_states = []
        for t in range(seq_len):
            h = self.forward_step(X[t], h)
            hidden_states.append(h.copy())
        return hidden_states

def demonstrate_vanishing_gradient_problem():
    """
    توضيح عملي لمشكلة تلاشي التدرجات في RNN: عبر قاعدة السلسلة،
    تدرج الخسارة بالنسبة لحالة مخفية بعيدة زمنيًا هو حاصل ضرب
    سلسلة طويلة من مشتقات tanh (أقصاها 1، وعادة أقل بكثير) مضروبة
    في W_hh تكراريًا. إذا كانت القيم الذاتية لـ W_hh أقل من 1،
    هذا الحاصل يتلاشى أُسّيًا مع طول التسلسل.
    """
    seq_lengths = [5, 20, 50, 100]
    # محاكاة مبسّطة: نفترض متوسط عامل انكماش لكل خطوة زمنية = 0.5
    # (قيمة واقعية جدًا لمشتقة tanh في أغلب نطاقات التشغيل العملية)
    decay_factor_per_step = 0.5

    print("تأثير طول التسلسل على مقدار التدرج المُتراكم (محاكاة):")
    for length in seq_lengths:
        gradient_magnitude = decay_factor_per_step ** length
        print(f"  طول التسلسل={length}: مقدار التدرج ≈ {gradient_magnitude:.2e}")

demonstrate_vanishing_gradient_problem()
# طول التسلسل=5: مقدار التدرج ≈ 3.13e-02
# طول التسلسل=20: مقدار التدرج ≈ 9.54e-07
# طول التسلسل=50: مقدار التدرج ≈ 8.88e-16
# طول التسلسل=100: مقدار التدرج ≈ 7.89e-31   <- تلاشى عمليًا تمامًا!


> 🔴 **هذا بالضبط سبب "النسيان" الذي تُعاني منه RNN البسيطة**: الكلمات الأولى في جملة أو فقرة طويلة لا تحصل عمليًا على أي تدرج تدريبي ذي معنى، فلا يتعلّم النموذج الاعتماديات طويلة المدى (Long-range dependencies) إطلاقًا.

### LSTM — الحل عبر البوابات (Gates)


In [ ]:
class LSTMCellFromScratch:
    """
    تطبيق كامل لخلية LSTM من الصفر (Hochreiter & Schmidhuber, 1997).
    الفكرة الثورية: فصل "حالة الخلية" (Cell State, C) عن "الحالة
    المخفية" (Hidden State, h). حالة الخلية تعمل كـ"حزام ناقل"
    (conveyor belt) يسمح للمعلومة بالتدفق عبر الزمن مع تدخل خطي
    بسيط فقط (جمع وضرب عنصري)، مما يحل مشكلة تلاشي التدرجات بشكل
    كبير جدًا مقارنة بـ RNN البسيطة.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        scale = 1.0 / np.sqrt(hidden_dim)

        # أربع بوابات، كل منها لها أوزانها الخاصة (تُطبَّق على [x_t, h_{t-1}] مُدمَجين)
        self.W_forget = np.random.randn(concat_dim, hidden_dim) * scale
        self.W_input = np.random.randn(concat_dim, hidden_dim) * scale
        self.W_candidate = np.random.randn(concat_dim, hidden_dim) * scale
        self.W_output = np.random.randn(concat_dim, hidden_dim) * scale

        self.b_forget = np.ones(hidden_dim)  # تهيئة forget bias بـ1 (حيلة مهمة!)
        self.b_input = np.zeros(hidden_dim)
        self.b_candidate = np.zeros(hidden_dim)
        self.b_output = np.zeros(hidden_dim)

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward_step(self, x_t: np.ndarray, h_prev: np.ndarray, c_prev: np.ndarray):
        """
        أربع بوابات، كل منها بدور محدد بدقة:

        1. Forget Gate (f_t): "كم من الذاكرة القديمة (c_prev) نحتفظ بها؟"
        2. Input Gate (i_t): "كم من المعلومة الجديدة ندخلها للذاكرة؟"
        3. Candidate (c̃_t): "ما هي المعلومة الجديدة المُرشَّحة للإضافة؟"
        4. Output Gate (o_t): "كم من الذاكرة الحالية نُظهِر كحالة مخفية؟"
        """
        combined = np.concatenate([x_t, h_prev])

        forget_gate = self.sigmoid(combined @ self.W_forget + self.b_forget)
        input_gate = self.sigmoid(combined @ self.W_input + self.b_input)
        candidate = np.tanh(combined @ self.W_candidate + self.b_candidate)
        output_gate = self.sigmoid(combined @ self.W_output + self.b_output)

        # تحديث حالة الخلية: مزيج من "نسيان جزء من القديم" و"إضافة جزء من الجديد"
        # هذا التحديث **خطي بشكل أساسي** (جمع + ضرب عنصري، لا tanh متكرر)
        # وهذا بالضبط ما يسمح للتدرج بالتدفق عبر مسافات زمنية طويلة دون تلاشٍ شديد
        c_t = forget_gate * c_prev + input_gate * candidate

        # الحالة المخفية: جزء "مُرشَّح للإظهار" من حالة الخلية الجديدة
        h_t = output_gate * np.tanh(c_t)

        return h_t, c_t, {
            "forget_gate": forget_gate, "input_gate": input_gate,
            "candidate": candidate, "output_gate": output_gate,
        }

    def forward_sequence(self, X: np.ndarray):
        seq_len = X.shape[0]
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        hidden_states, cell_states = [], []

        for t in range(seq_len):
            h, c, gates = self.forward_step(X[t], h, c)
            hidden_states.append(h.copy())
            cell_states.append(c.copy())

        return hidden_states, cell_states


> 💡 **لماذا تُهيَّأ `b_forget` بالقيمة 1 (وليس صفر)؟** هذه حيلة هندسية مهمة جدًا ومُثبَتة تجريبيًا (Jozefowicz et al., 2015): في بداية التدريب، تُهيَّأ بوابة النسيان لتكون "منفتحة" (قريبة من 1، أي "احتفظ بمعظم الذاكرة القديمة") بدلًا من عشوائية تمامًا، مما يسمح للتدرجات بالتدفق بحرية أكبر عبر الزمن منذ الخطوات الأولى من التدريب ويُسرِّع التقارب بشكل ملحوظ.

### GRU — نسخة مُبسَّطة من LSTM


In [ ]:
class GRUCellFromScratch:
    """
    تطبيق GRU (Cho et al., 2014) من الصفر: دمج بوابتي Forget وInput
    في بوابة واحدة (Update Gate)، وإزالة حالة الخلية المنفصلة تمامًا
    (الاعتماد فقط على الحالة المخفية h). نتيجة ذلك: معاملات أقل
    بنسبة ~25% من LSTM، وتدريب أسرع، مع أداء مقارب جدًا في أغلب المهام.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim
        scale = 1.0 / np.sqrt(hidden_dim)

        self.W_update = np.random.randn(concat_dim, hidden_dim) * scale
        self.W_reset = np.random.randn(concat_dim, hidden_dim) * scale
        self.W_candidate = np.random.randn(concat_dim, hidden_dim) * scale

        self.b_update = np.zeros(hidden_dim)
        self.b_reset = np.zeros(hidden_dim)
        self.b_candidate = np.zeros(hidden_dim)

    @staticmethod
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def forward_step(self, x_t: np.ndarray, h_prev: np.ndarray):
        combined = np.concatenate([x_t, h_prev])

        # Update Gate: يُحدِّد "مزيج" الاحتفاظ بالقديم مقابل الجديد (يدمج دوري forget+input من LSTM)
        update_gate = self.sigmoid(combined @ self.W_update + self.b_update)

        # Reset Gate: يُحدِّد "كم من الماضي نتجاهل" عند حساب المُرشَّح الجديد
        reset_gate = self.sigmoid(combined @ self.W_reset + self.b_reset)

        combined_reset = np.concatenate([x_t, reset_gate * h_prev])
        candidate = np.tanh(combined_reset @ self.W_candidate + self.b_candidate)

        # التحديث النهائي: استيفاء خطي بين الحالة القديمة والمُرشَّح الجديد
        h_t = (1 - update_gate) * h_prev + update_gate * candidate
        return h_t


### الانتقال لـ PyTorch ومعمارية Bidirectional


In [ ]:
import torch
import torch.nn as nn

class LSTMTextClassifier(nn.Module):
    """
    مُصنِّف نصوص كامل باستخدام LSTM ثنائي الاتجاه (Bidirectional):
    يُعالِج التسلسل من البداية للنهاية ومن النهاية للبداية في آن واحد،
    مما يسمح لكل موضع بالوصول لمعلومات من كلا الاتجاهين — تحسين
    مهم جدًا قبل ظهور Attention، لكنه لا يزال أبطأ وأقل فعالية
    من Self-Attention الكامل (الفصل 10) في التقاط الاعتماديات
    البعيدة جدًا.
    """
    def __init__(self, vocab_size: int, embedding_dim: int = 128,
                 hidden_dim: int = 256, num_classes: int = 2,
                 num_layers: int = 2, dropout: float = 0.3,
                 pad_idx: int = 0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,     # معالجة في الاتجاهين
            dropout=dropout if num_layers > 1 else 0,
        )

        # ×2 لأن bidirectional يُركِّب (concatenate) الاتجاهين معًا
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor = None):
        embedded = self.embedding(input_ids)   # (batch, seq_len, embedding_dim)

        if lengths is not None:
            # Packing: تحسين كفاءة حسابية مهم جدًا عند وجود padding
            # (يتجاهل الحسابات على مواضع الحشو، يُسرّع التدريب بشكل ملحوظ)
            packed = nn.utils.rnn.pack_padded_sequence(
                embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            packed_output, (h_n, c_n) = self.lstm(packed)
        else:
            _, (h_n, c_n) = self.lstm(embedded)

        # h_n شكلها: (num_layers*2, batch, hidden_dim)
        # نأخذ الحالة المخفية الأخيرة من كلا الاتجاهين للطبقة الأخيرة فقط
        forward_last = h_n[-2, :, :]   # آخر حالة للاتجاه الأمامي
        backward_last = h_n[-1, :, :]  # آخر حالة للاتجاه الخلفي
        combined = torch.cat([forward_last, backward_last], dim=1)

        combined = self.dropout(combined)
        logits = self.fc(combined)
        return logits


### أبرز قيود RNN/LSTM التي حفّزت اختراع المحوّلات


In [ ]:
def summarize_rnn_limitations():
    """
    ملخص القيود الجوهرية التي دفعت الباحثين لاختراع Attention
    والمحوّلات بالكامل (سنرى الحل الكامل في الفصل 10).
    """
    limitations = {
        "المعالجة المتسلسلة الإلزامية": (
            "لا يمكن حساب h_t دون معرفة h_{t-1} أولًا، مما يمنع "
            "التوازي الكامل عبر طول التسلسل أثناء التدريب — عنق "
            "زجاجة حاسوبي خطير جدًا لتدريب نماذج كبيرة على GPUs حديثة "
            "مُصمَّمة أساسًا للتوازي الضخم."
        ),
        "صعوبة الاعتماديات طويلة المدى": (
            "رغم تحسّن LSTM/GRU الكبير عن RNN البسيطة، لا تزال المعلومة "
            "تمر عبر 'عنق زجاجة' حالة مخفية ثابتة الحجم، ما يُصعِّب "
            "الاحتفاظ بتفاصيل دقيقة من بداية تسلسل طويل جدًا (آلاف الكلمات)."
        ),
        "صعوبة بناء 'خريطة اعتماديات' مباشرة وصريحة": (
            "لا توجد آلية مباشرة لأي كلمة للوصول مباشرة (بخطوة واحدة) "
            "لأي كلمة أخرى بعيدة في التسلسل — كل المعلومة يجب أن تمر "
            "عبر سلسلة من التحديثات التسلسلية الوسيطة."
        ),
    }

    for problem, explanation in limitations.items():
        print(f"❌ {problem}:\n   {explanation}\n")

summarize_rnn_limitations()


---


## 4. مثال محلول خطوة بخطوة

**المسألة**: قارن عدد المعاملات القابلة للتدريب بين RNN بسيطة وLSTM وGRU لنفس الأبعاد (input_dim=100, hidden_dim=256)، لفهم تكلفة كل معمارية.


In [ ]:
def compare_parameter_counts(input_dim=100, hidden_dim=256):
    # RNN بسيطة: W_xh + W_hh + b_h
    rnn_params = (input_dim * hidden_dim) + (hidden_dim * hidden_dim) + hidden_dim

    # LSTM: 4 بوابات، كل منها بنفس حجم أوزان RNN تقريبًا
    concat_dim = input_dim + hidden_dim
    lstm_params = 4 * (concat_dim * hidden_dim + hidden_dim)

    # GRU: 3 "بوابات" فقط (update, reset, candidate)
    gru_params = 3 * (concat_dim * hidden_dim + hidden_dim)

    print(f"RNN البسيطة:  {rnn_params:,} معامل")
    print(f"LSTM:         {lstm_params:,} معامل ({lstm_params/rnn_params:.1f}× من RNN)")
    print(f"GRU:          {gru_params:,} معامل ({gru_params/rnn_params:.1f}× من RNN، "
          f"{gru_params/lstm_params:.1%} من LSTM)")

compare_parameter_counts()
# RNN البسيطة:  91,392 معامل
# LSTM:         365,568 معامل (4.0× من RNN)
# GRU:          274,176 معامل (3.0× من RNN، 75.0% من LSTM)


---


## 5. نصائح وحيل

| النصيحة | التفصيل |
|---|---|
| استخدم `pack_padded_sequence` دائمًا مع تسلسلات متفاوتة الطول | يُسرِّع التدريب بشكل ملحوظ ويمنع تأثير الـ padding على التدرجات |
| ابدأ بـ GRU قبل LSTM إذا كانت الموارد محدودة | أداء مقارب جدًا بمعاملات أقل وتدريب أسرع |
| استخدم Bidirectional دائمًا للتصنيف (غير مناسب للتوليد) | يُحسِّن الأداء بشكل ملحوظ عندما يكون التسلسل الكامل متاحًا مسبقًا (ليس توليدًا تتابعيًا) |
| راقب تدرجات طبقات RNN المبكرة أثناء التدريب | قيم قريبة من الصفر باستمرار علامة على تلاشي التدرجات رغم استخدام LSTM/GRU |


## 6. أخطاء شائعة ومصائد

- **استخدام RNN البسيطة لتسلسلات طويلة**: تفشل عمليًا بعد ~10-20 خطوة زمنية بسبب تلاشي التدرجات الشديد كما أثبتنا رياضيًا وعدديًا أعلاه.
- **نسيان Gradient Clipping مع RNN/LSTM**: هذه المعماريات عرضة بشكل خاص لمشكلة "انفجار التدرجات" (Exploding Gradients) المعاكسة لمشكلة التلاشي، خصوصًا مع تسلسلات طويلة.
- **مقارنة RNN بالمحوّلات دون فهم السياق التاريخي**: RNN لم تكن "خطأ"، بل كانت أفضل حل متاح في زمانها — فهم قيودها الدقيقة هو ما يُقدِّر جمال حل Attention لاحقًا.


## 7. جدول مقارنة

| الخاصية | RNN البسيطة | LSTM | GRU |
|---|---|---|---|
| عدد البوابات | 0 | 4 (forget, input, output, candidate) | 3 (update, reset, candidate) |
| مشكلة تلاشي التدرجات | شديدة جدًا | محلولة جزئيًا بشكل جيد | محلولة جزئيًا بشكل جيد |
| عدد المعاملات | الأقل | الأعلى | متوسط (~75% من LSTM) |
| سرعة التدريب | الأسرع (نظريًا) لكن أداء ضعيف | الأبطأ | أسرع من LSTM بأداء مقارب |
| الاستخدام الحديث | نادر جدًا، تعليمي فقط | لا يزال مستخدَمًا في تطبيقات متخصصة | بديل خفيف لـ LSTM |


## 8. ملخص أفضل الممارسات

- افهم RNN البسيطة ومشكلة تلاشي التدرجات رياضيًا بعمق — هذا يُؤسِّس فهمك لكل ما يليها.
- استخدم LSTM/GRU فقط لمهام تسلسلية بسيطة اليوم (المحوّلات هي المعيار لكل شيء آخر تقريبًا).
- طبّق `pack_padded_sequence` وGradient Clipping دائمًا عند استخدام RNN/LSTM/GRU فعليًا.
- احتفظ بفهم هذه المعماريات التاريخية جيدًا — بعض المعماريات الحديثة (Mamba، الفصل 16) تستعير مبادئ مشابهة للحالة المتكررة بكفاءة حديثة جديدة.


## 9. أسئلة مقابلات

**س: كيف تحل LSTM مشكلة تلاشي التدرجات التي تُعاني منها RNN البسيطة؟**

**ج**: المفتاح هو فصل "حالة الخلية" (Cell State، C_t) عن الحالة المخفية، وتحديثها عبر عملية **خطية بشكل أساسي**: `C_t = forget_gate ⊙ C_{t-1} + input_gate ⊙ candidate`. هذا التحديث يتضمّن فقط جمعًا وضربًا عنصريًا (Hadamard product)، وليس ضربًا مصفوفيًا متكررًا عبر `tanh` كما في RNN البسيطة. عند اشتقاق التدرج عبر الزمن (Backpropagation Through Time)، هذا يعني أن مسار التدرج عبر حالة الخلية يمر بشكل أساسي عبر عمليات `forget_gate` (قيمة بين 0 و1، لكنها **مُتعلَّمة ومُتكيّفة** حسب المدخل الفعلي)، بدلًا من الضرب المتكرر بنفس مصفوفة الأوزان `W_hh` ومشتقة `tanh` الثابتة نسبيًا كما في RNN. عندما تتعلّم بوابة النسيان أن تبقي قيمتها قريبة من 1 لمعلومة مهمة، يمر التدرج عبر هذا المسار بأقل انكماش ممكن، مما يسمح للشبكة بتعلّم الاحتفاظ بمعلومات لمسافات زمنية أطول بكثير مقارنة بـ RNN البسيطة.

**س: لماذا تُعدّ المعالجة المتسلسلة الإلزامية في RNN عيبًا جوهريًا في عصر GPUs الحديثة، حتى لو كان أداء النموذج جيدًا؟**

**ج**: GPUs الحديثة مُصمَّمة أساسًا للتوازي الضخم (آلاف النوى تعمل في آن واحد). في RNN، حساب `h_t` يعتمد بشكل إلزامي على `h_{t-1}` المحسوبة بالفعل، مما يعني أنه **لا يمكن حساب خطوتين زمنيتين متتاليتين في آن واحد** بغض النظر عن توفر موارد حوسبية متوازية هائلة — التسلسل بطبيعته الرياضية "متسلسل" (sequential dependency). هذا يعني أن وقت التدريب يتناسب خطيًا تقريبًا مع طول التسلسل بغض النظر عن حجم الـ GPU المستخدَم، بعكس المحوّلات (الفصل 10) حيث يمكن حساب Attention لكل المواضع في التسلسل **في آن واحد بشكل متوازٍ بالكامل** (نفس الضرب المصفوفي الكبير يُغطّي كل المواضع معًا)، مما يستغل قدرة GPUs الحديثة استغلالًا كاملًا ويُقلِّص زمن التدريب الفعلي بشكل جذري لمجموعات بيانات وتسلسلات ضخمة.


## 10. تمرين عملي

نفّذ `LSTMCellFromScratch` و`GRUCellFromScratch` كاملتين، وطبّقهما على نفس تسلسل بيانات صناعي طويل (100+ خطوة زمنية) حيث المعلومة المهمة موجودة في أول 5 خطوات فقط ويجب تذكّرها للتنبؤ بالمخرج النهائي. قارن قدرة كل من RNN البسيطة وLSTM وGRU على حل هذه المهمة (ستفشل RNN البسيطة بوضوح شديد).


## 11. مراجع للتعمّق

- Hochreiter & Schmidhuber (1997) — *Long Short-Term Memory*
- Cho et al. (2014) — *Learning Phrase Representations using RNN Encoder-Decoder* (GRU)
- Bengio, Simard & Frasconi (1994) — *Learning Long-Term Dependencies with Gradient Descent is Difficult*
- Jozefowicz, Zaremba & Sutskever (2015) — *An Empirical Exploration of Recurrent Network Architectures*


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — التمثيل اللغوي / الفصل 8: الشبكات المتكررة: RNN وLSTM وGRU.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
